# PATHSNAP: Dataset Exploration & Integrity Audit
## Distributed Map-Matching & Route Reconstruction

This notebook performs comprehensive exploratory data analysis (EDA) on the raw PATHSNAP dataset.


### 1. Environment Setup & Configuration


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Set up styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

with open('../config/preprocessing_config.yaml', 'r') as f:
    config = yaml.safe_load(f)
print("Config loaded successfully!")


### 2. Dataset Inventory Audit
Let's load the generated dataset inventory report containing all 90 inspected trips across train and student_test partitions.


In [ ]:
inventory_df = pd.read_csv('../outputs/reports/dataset_inventory.csv')
print(f"Total trips inspected: {len(inventory_df)}")
print(f"Partitions: {inventory_df['partition'].value_counts().to_dict()}")
inventory_df.head(10)[['trip_id', 'partition', 'track_rows', 'nodes_rows', 'arcs_rows', 'route_rows', 'track_dt_mean']]


### 3. File Formats & Preserving 0-Indexed IDs
Recall the critical structural requirement:
- `.nodes` row indices represent node IDs (0 to N-1).
- `.arcs` row indices represent arc IDs (0 to M-1).
- Never sort or delete node/arc rows.


In [ ]:
# Inspect sample raw trip
from src.data_loader import load_trip

sample_trip = load_trip('../data/raw/train/00000001', '00000001', 'train')
print("Track sample (GPS observations):")
print(sample_trip.track.head())
print("
Nodes sample (0-indexed node IDs preserved):")
print(sample_trip.nodes.head())
print("
Arcs sample (0-indexed arc IDs preserved):")
print(sample_trip.arcs.head())
print("
Reference route sample (chronological arc IDs):")
print(sample_trip.route.head())


### 4. Global Geographic Distribution & Metric Projection
Because trips come from different regions worldwide, each trip uses its localized UTM projection.


In [ ]:
from src.coordinate_utils import get_utm_epsg, CoordinateTransformer

for trip_id in ['00000001', '00000008', '00000010', '00000011']:
    row = inventory_df[inventory_df['trip_id'] == int(trip_id)].iloc[0]
    center_lon = (row['track_min_lon'] + row['track_max_lon']) / 2
    center_lat = (row['track_min_lat'] + row['track_max_lat']) / 2
    epsg = get_utm_epsg(center_lon, center_lat)
    print(f"Trip {trip_id}: Center=({center_lon:.2f}, {center_lat:.2f}) -> UTM EPSG:{epsg}")


### 5. Road Network Topology & Connected Components


In [ ]:
val_df = pd.read_csv('../outputs/reports/data_quality_report.csv')
print("Summary of Graph Connectivity:")
print(val_df[['wcc_count', 'scc_count', 'largest_wcc_ratio', 'arc_duplicate_count']].describe())
